# 01 — Data Pipeline

Runs the full data pipeline for **AirfRANS** and **CFDBench**:
1. Load session config from Drive (written by `00_env_setup.ipynb`)
2. Download raw datasets
3. Convert to unified HDF5 schema
4. Embed dimensionless numbers per case
5. Validate output + print summary stats

**Prerequisite:** Run `00_env_setup.ipynb` first to mount Drive and install dependencies.

**Estimated runtime (A100 Colab):**
- AirfRANS full (1000 cases): ~25 min download + ~10 min convert
- CFDBench (all 4 problems): ~15 min download + ~8 min convert

## 0. Load session config + paths

In [ ]:
import json, os, sys
from pathlib import Path

# Mount Drive if not already mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# Load session config
BASE_DIR = '/content/drive/MyDrive/cfd-ald-app' if ON_COLAB else '.'
cfg_path = Path(BASE_DIR) / 'session_config.json'

if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    REPO_DIR = cfg['repo_dir']
    print(f'Session config loaded: {cfg}')
else:
    # Fallback for local Mac dev
    REPO_DIR = str(Path().resolve())
    print(f'No session_config.json found — using local path: {REPO_DIR}')

sys.path.insert(0, REPO_DIR)

# Data paths
RAW_DIR       = Path(BASE_DIR) / 'data' / 'raw'
PROCESSED_DIR = Path(BASE_DIR) / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data     → {RAW_DIR}')
print(f'Processed    → {PROCESSED_DIR}')

## 1. Install dataset-specific packages

These are lightweight and fast to install.

In [ ]:
AIRFRANS_TASK = 'scarce'   # 'scarce' (200 cases) or 'full' (800 cases)
AIRFRANS_RAW  = str(RAW_DIR / 'airfrans')

manifest = Path(AIRFRANS_RAW) / 'manifest.json'
if manifest.exists():
    existing = json.loads(manifest.read_text())
    print(f'AirfRANS already downloaded: {len(existing)} cases. Skipping.')
else:
    !python3 {REPO_DIR}/data/scripts/download_airfrans.py --out {AIRFRANS_RAW} --task {AIRFRANS_TASK}

## 2. Download AirfRANS

**`scarce` split** = 200 training cases (fast, good for initial surrogate training)  
**`full` split**   = 1000 cases (use for final training runs)

Set `AIRFRANS_SPLIT` below.

In [ ]:
AIRFRANS_SPLIT = 'scarce'   # 'scarce' (200 cases) or 'full' (1000 cases)
AIRFRANS_RAW   = str(RAW_DIR / 'airfrans')

# Skip if already downloaded
manifest = Path(AIRFRANS_RAW) / 'manifest.json'
if manifest.exists():
    existing = json.loads(manifest.read_text())
    print(f'AirfRANS already downloaded: {len(existing)} cases. Skipping.')
else:
    !python3 {REPO_DIR}/data/scripts/download_airfrans.py \
        --out {AIRFRANS_RAW} \
        --splits {AIRFRANS_SPLIT}

## 3. Download CFDBench

In [ ]:
CFDBENCH_RAW = str(RAW_DIR / 'cfdbench')

manifest = Path(CFDBENCH_RAW) / 'manifest.json'
if manifest.exists():
    existing = json.loads(manifest.read_text())
    print(f'CFDBench already downloaded: {len(existing)} cases. Skipping.')
else:
    !python3 {REPO_DIR}/data/scripts/download_cfdbench.py \
        --out {CFDBENCH_RAW} \
        --problems cavity tube dam cylinder

## 4. Convert AirfRANS → HDF5 (pointcloud schema)

In [ ]:
AIRFRANS_PROC = str(PROCESSED_DIR / 'airfrans')

case_index = Path(AIRFRANS_PROC) / 'case_index.json'
if case_index.exists():
    n = len(json.loads(case_index.read_text()))
    print(f'AirfRANS already converted: {n} HDF5 cases. Skipping.')
else:
    !python3 {REPO_DIR}/data/scripts/convert_to_hdf5.py airfrans \
        --raw {AIRFRANS_RAW} \
        --out {AIRFRANS_PROC}

## 5. Convert CFDBench → HDF5 (grid schema)

In [ ]:
CFDBENCH_PROC = str(PROCESSED_DIR / 'cfdbench')

case_index = Path(CFDBENCH_PROC) / 'case_index.json'
if case_index.exists():
    n = len(json.loads(case_index.read_text()))
    print(f'CFDBench already converted: {n} HDF5 cases. Skipping.')
else:
    !python3 {REPO_DIR}/data/scripts/convert_to_hdf5.py cfdbench \
        --raw {CFDBENCH_RAW} \
        --out {CFDBENCH_PROC}

## 6. Embed dimensionless numbers

In [ ]:
!python3 {REPO_DIR}/data/scripts/compute_dimensionless.py \
    --processed {AIRFRANS_PROC} --dataset airfrans

!python3 {REPO_DIR}/data/scripts/compute_dimensionless.py \
    --processed {CFDBENCH_PROC} --dataset cfdbench

## 7. Validation + summary stats

In [ ]:
import h5py
import numpy as np
from pathlib import Path

def validate_dataset(proc_dir: str, representation: str, n_sample: int = 5):
    proc = Path(proc_dir)
    h5_files = sorted(proc.rglob('*.h5'))[:n_sample]

    if not h5_files:
        print(f'  ✗  No HDF5 files found in {proc_dir}')
        return

    print(f'\n── {proc.name} ({representation}) ──')
    meta_path = proc / 'metadata.json'
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        print(f'  Fields:       {meta["fields"]}')
        print(f'  Dimensionless:{meta["dimensionless"]}')

    total = len(list(proc.rglob('*.h5')))
    print(f'  Total cases:  {total}')

    # Spot-check first file
    with h5py.File(h5_files[0], 'r') as h5:
        print(f'  Sample file:  {h5_files[0].name}')
        for key in sorted(h5.keys()):
            for subkey in sorted(h5[key].keys()) if hasattr(h5[key], 'keys') else []:
                ds = h5[f'{key}/{subkey}']
                print(f'    /{key}/{subkey}: shape={ds.shape}  dtype={ds.dtype}')

        g = h5['inputs/global'][:]
        g_cols = list(h5['inputs/global'].attrs.get('columns', []))
        print(f'  Global feats: {dict(zip(g_cols, g.round(4)))}')

        # Check Re is in a plausible range
        if 'Re' in g_cols:
            Re = g[g_cols.index('Re')]
            assert Re > 0, f'Re should be positive, got {Re}'
            print(f'  Re check: {Re:.1f} ✓')

    print(f'  ✓  {proc.name} looks healthy')


validate_dataset(AIRFRANS_PROC, 'pointcloud')
validate_dataset(CFDBENCH_PROC, 'grid')

## 8. Re distribution plots

Quick sanity check: AirfRANS should span Re ~ 1M–6M (chord-based, airfoil flow).  
CFDBench should show lower Re for laminar cavity/tube cases.

In [ ]:
import matplotlib.pyplot as plt

def collect_global_feature(proc_dir: str, feature: str, max_files: int = 200):
    values = []
    for h5_path in sorted(Path(proc_dir).rglob('*.h5'))[:max_files]:
        with h5py.File(h5_path, 'r') as h5:
            g_cols = list(h5['inputs/global'].attrs.get('columns', []))
            if feature in g_cols:
                g = h5['inputs/global'][:]
                values.append(float(g[g_cols.index(feature)]))
    return np.array(values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

re_airfrans = collect_global_feature(AIRFRANS_PROC, 'Re')
if len(re_airfrans):
    axes[0].hist(re_airfrans, bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title(f'AirfRANS — Re distribution (n={len(re_airfrans)})')
    axes[0].set_xlabel('Re'); axes[0].set_ylabel('Count')

re_cfdbench = collect_global_feature(CFDBENCH_PROC, 'Re')
if len(re_cfdbench):
    axes[1].hist(re_cfdbench, bins=30, color='darkorange', edgecolor='white')
    axes[1].set_title(f'CFDBench — Re distribution (n={len(re_cfdbench)})')
    axes[1].set_xlabel('Re'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / 'Re_distributions.png'), dpi=120)
plt.show()
print('Plot saved to processed/Re_distributions.png')

## 9. Save pipeline summary

Records how many cases were processed so training notebooks can assert the data is ready.

In [ ]:
summary = {
    'airfrans': {
        'n_cases':  len(list(Path(AIRFRANS_PROC).rglob('*.h5'))),
        'proc_dir': AIRFRANS_PROC,
    },
    'cfdbench': {
        'n_cases':  len(list(Path(CFDBENCH_PROC).rglob('*.h5'))),
        'proc_dir': CFDBENCH_PROC,
    },
}

summary_path = Path(BASE_DIR) / 'data_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f'\nSummary saved to {summary_path}')
print('\nData pipeline complete ✓  — ready for 01_pretrain_flow.ipynb')